In [1]:
# Import and setup
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
#create a cache folder -  FastF1 saves downloads data here
# so it dosen't re-download everytime we run the code
cache_path = '../data/raw/fastf1_cache'
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)


print("✅ All libraries imported successfully!")
print("FastF1 version:", fastf1.__version__)

✅ All libraries imported successfully!
FastF1 version: 3.8.1


In [3]:
# Cell 2 — Load a Real F1 Race Session
# We are loading the 2023 British Grand Prix Race
# fastf1.get_session() needs 3 things:
#   1. Year — which F1 season
#   2. Event name — which Grand Prix
#   3. Session type — 'R' means Race

session = fastf1.get_session(2023, 'British Grand Prix', 'R')

# .load() downloads the actual data from F1's servers
# This may take 1-2 minutes the first time — it's downloading real race data!
# After that it loads from your cache folder instantly
session.load(laps=True, telemetry=True, weather=True)

print("✅ Session loaded!")
print("Event:", session.event['EventName'])
print("Date:", session.date)
print("Total laps in dataset:", len(session.laps))

core           INFO 	Loading data for British Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca

✅ Session loaded!
Event: British Grand Prix
Date: 2023-07-09 14:00:00
Total laps in dataset: 971


In [4]:
# session.laps gives us a table of ALL laps from ALL drivers
# Think of it like an Excel spreadsheet with 971 rows

laps = session.laps

# .shape tells us (number of rows, number of columns)
print("Shape:", laps.shape)

# Let's see what data columns we have available
print("\nAvailable columns:")
print(laps.columns.tolist())

# .head() shows the first 5 rows so we can see what it looks like
laps.head()

Shape: (971, 31)

Available columns:
['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate']


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 01:03:46.121000,VER,1,0 days 00:01:37.167000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:38.051000,...,True,Red Bull Racing,0 days 01:02:08.731000,2023-07-09 14:03:09.767,1,2.0,False,,False,False
1,0 days 01:05:19.554000,VER,1,0 days 00:01:33.433000,2.0,1.0,NaT,NaT,0 days 00:00:29.616000,0 days 00:00:38,...,True,Red Bull Racing,0 days 01:03:46.121000,2023-07-09 14:04:47.157,1,2.0,False,,False,True
2,0 days 01:06:52.284000,VER,1,0 days 00:01:32.730000,3.0,1.0,NaT,NaT,0 days 00:00:29.380000,0 days 00:00:37.690000,...,True,Red Bull Racing,0 days 01:05:19.554000,2023-07-09 14:06:20.590,1,2.0,False,,False,True
3,0 days 01:08:25.064000,VER,1,0 days 00:01:32.780000,4.0,1.0,NaT,NaT,0 days 00:00:29.407000,0 days 00:00:37.650000,...,True,Red Bull Racing,0 days 01:06:52.284000,2023-07-09 14:07:53.320,1,2.0,False,,False,True
4,0 days 01:09:57.646000,VER,1,0 days 00:01:32.582000,5.0,1.0,NaT,NaT,0 days 00:00:29.338000,0 days 00:00:37.403000,...,True,Red Bull Racing,0 days 01:08:25.064000,2023-07-09 14:09:26.100,1,1.0,False,,False,True


In [5]:
# Cell 4 — Clean and Prepare the Data
# We pick only the columns we need and remove bad laps

# These are the columns most useful for our ML model later
columns_we_need = [
    'Driver',        # 3-letter driver code e.g. VER, HAM
    'Team',          # Constructor name e.g. Red Bull Racing
    'LapNumber',     # Which lap of the race (1 to 52)
    'LapTime',       # How long the lap took as a duration
    'Stint',         # 1 = before first pitstop, 2 = after, etc
    'TyreLife',      # How many laps this tyre set has been used
    'Compound',      # Tyre type: SOFT, MEDIUM, HARD
    'PitInTime',     # Time driver entered pit lane (NaT = no pit stop)
    'PitOutTime',    # Time driver left pit lane (NaT = no pit stop)
    'SpeedI1',       # Speed at first speed measurement point (km/h)
    'SpeedI2',       # Speed at second speed measurement point (km/h)
    'SpeedFL',       # Speed at finish line (km/h)
    'SpeedST',       # Speed in speed trap (km/h)
    'IsAccurate',    # FastF1 quality flag — True means data is reliable
]

# Copy only the columns we need into a new DataFrame
df = laps[columns_we_need].copy()

# Convert LapTime from duration format to plain seconds
# e.g. 00:01:32.730 becomes 92.730
# .dt.total_seconds() extracts the total number of seconds
df['LapTimeSeconds'] = df['LapTime'].dt.total_seconds()

# Remove laps FastF1 has flagged as unreliable
df = df[df['IsAccurate'] == True]

# Remove laps with no lap time recorded
df = df.dropna(subset=['LapTimeSeconds'])

# Remove outlier laps — safety car laps are very slow (over 120s)
# First lap is chaotic so we skip it too
df = df[(df['LapTimeSeconds'] > 75) & (df['LapTimeSeconds'] < 120)]
df = df[df['LapNumber'] > 1]

print(f"✅ Clean laps: {len(df)} rows")
print(f"Drivers: {df['Driver'].nunique()}")
print(f"\nTyre compounds used:")
print(df['Compound'].value_counts())
print(f"\nLap time range: {df['LapTimeSeconds'].min():.2f}s — {df['LapTimeSeconds'].max():.2f}s")

✅ Clean laps: 802 rows
Drivers: 20

Tyre compounds used:
Compound
MEDIUM    438
SOFT      248
HARD      116
Name: count, dtype: int64

Lap time range: 90.28s — 96.74s


In [6]:
# Save Clean Data to CSV
# We save it so other notebooks can use it without re-downloading

# Create the output folder if it doesn't exist
# exist_ok=True means no error if folder already exists
os.makedirs('../data/processed', exist_ok=True)

output_path = '../data/processed/british_gp_2023_clean.csv'

# index=False means don't save the row numbers (0,1,2,3...)
# we don't need them and they just clutter the file
df.to_csv(output_path, index=False)

print(f"✅ Data saved to {output_path}")
print(f"Rows saved: {len(df)}")
print(f"Columns saved: {df.columns.tolist()}")

✅ Data saved to ../data/processed/british_gp_2023_clean.csv
Rows saved: 802
Columns saved: ['Driver', 'Team', 'LapNumber', 'LapTime', 'Stint', 'TyreLife', 'Compound', 'PitInTime', 'PitOutTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsAccurate', 'LapTimeSeconds']
